# 00b — Split Bersama dengan Penanganan Teks Duplikat

Di dataset ada teks yang **isinya sama tapi `text_id`-nya berbeda** (teks yang sama dianotasi ulang di batch lain).
Pada `00_make_splits.ipynb`, split dilakukan per `text_id`, sehingga salinan teks yang sama bisa masuk ke **train dan test sekaligus** (data leakage).

Notebook ini memperbaikinya dengan **split berbasis kelompok**:

1. Teks yang kembar diberi **ID kelompok** yang sama.
2. Split dilakukan **per kelompok**, sehingga semua salinan selalu berada di split yang sama.
3. Label dan data annotator **tidak diubah**. Yang dihapus hanya salinan yang benar-benar identik
   (teks, annotator, dan labelnya sama), karena tidak menambah informasi.

Output: `splits_grouped.csv` (format sama dengan `splits.csv`, ditambah kolom `group_id`).
Untuk memakainya, ganti `SPLIT_FILE` di notebook training menjadi file ini.


In [17]:
! pip install -q iterative-stratification emoji

In [18]:
import ast
import re
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

DATA_DIR       = "/content"                 # Colab/Kaggle: folder dataset
PREPROCESS_DIR = "/content"  # folder berisi preprocess.py
DATA_FILE      = f"{DATA_DIR}/data_thesis_full.csv"
OLD_SPLIT_FILE = f"{DATA_DIR}/splits.csv"         # split lama, hanya untuk perbandingan
OUT_FILE       = f"{DATA_DIR}/splits_grouped.csv"
SEED = 42

LABEL_COLS = ["insults", "identity_attack", "threat_incitement_to_violence",
              "profanity_obscenity", "sexually_explicit"]

from preprocess import preprocess_text

In [30]:
df = pd.read_csv(DATA_FILE, dtype={"text_id": str})

# jumlah annotator per teks (dihitung sebelum label di-vote)
df["n_annotators"] = df["toxicity"].apply(lambda x: len(ast.literal_eval(x)))

def majority_vote(x):
    votes = [int(v) for v in ast.literal_eval(x)]
    return int(np.mean(votes) > 0.5)

for col in ["is_noise_or_spam_text", "toxicity"] + LABEL_COLS:
    df[col] = df[col].apply(majority_vote)

df = df[df["is_noise_or_spam_text"] == 0]
df = df.dropna().reset_index(drop=True)
print("Jumlah teks:", len(df))
print(df["toxicity"].value_counts())

Jumlah teks: 26379
toxicity
0    23980
1     2399
Name: count, dtype: int64


## Buat kunci pembanding teks

Dua teks dianggap **kembar** jika salah satu kondisi ini terpenuhi:

- **Teks aslinya sama** setelah diabaikan huruf besar/kecil, mention, URL, angka, spasi, dan tanda baca.
- **Input modelnya sama**, yaitu hasil `preprocess_text` dari teks normalisasi LLM.


In [31]:
df["kunci_asli"]  = df["text"].apply(preprocess_text)
df["input_model"] = df["cleaned_text"].apply(preprocess_text)

df[["text_id", "text", "kunci_asli", "input_model"]].head(3)

,text_id,text,kunci_asli,input_model
0,1-1,Wajah Jurnalis Dibalut Perban Saat Melaporkan Kondisi di Gaza 2023,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza,wajah jurnalis dibalut perban saat melaporkan kondisi di gaza
1,1-10,"Elektabilitas Paslon 02 sebentar lagi terjun bebas gara2 Gemoy, jo...","elektabilitas paslon sebentar lagi terjun bebas gara gemoy, joget,...",elektabilitas pasangan calon sebentar lagi terjun bebas gara gara ...
2,1-100,Gini aja deh @KPU_ID usul aja nih. Gimana kalo misalnya gak usah a...,gini aja deh usul aja nih. gimana kalo misalnya gak usah ada debat...,"gini aja deh , usul aja nih. gimana kalau misalnya tidak usah ada ..."


## 4. Beri ID kelompok
Mulai dengan kelompok berdasarkan teks asli, lalu gabungkan kelompok yang input modelnya sama.
Diulang sampai tidak ada perubahan, karena penggabungan bisa berantai (A=B lewat teks asli, B=C lewat input model).

In [32]:
df["group_id"] = df.groupby("kunci_asli").ngroup()

while True:
    lama = df["group_id"].copy()
    df["group_id"] = df.groupby("input_model")["group_id"].transform("min")   # input model sama -> satu kelompok
    df["group_id"] = df.groupby("kunci_asli")["group_id"].transform("min")    # teks asli sama -> satu kelompok
    if (df["group_id"] == lama).all():
        break

ukuran = df.groupby("group_id")["text_id"].transform("size")
print("Jumlah teks             :", len(df))
print("Jumlah kelompok         :", df["group_id"].nunique())
print("Teks yang punya kembaran:", (ukuran > 1).sum())
print("Kelompok kembar         :", df.loc[ukuran > 1, "group_id"].nunique())

Jumlah teks             : 26379
Jumlah kelompok         : 24203
Teks yang punya kembaran: 4174
Kelompok kembar         : 1998


## Contoh teks kelompok kembar

In [33]:
contoh = df[ukuran > 1].sort_values("group_id")
for gid in contoh["group_id"].unique()[:3]:
    g = contoh[contoh["group_id"] == gid]
    print("=" * 90)
    print("Teks :", g["text"].iloc[0][:100])
    print(g[["text_id", "n_annotators", "toxicity"]].to_string(index=False))

Teks : ! Yuk kita dukung program penuntasan buta aksara yang digalakkan oleh Kemdikbudristek. Mari bantu me
text_id  n_annotators  toxicity
 3-1196             1         0
 2-6290             1         0
Teks : , dua China memandang Uyghur itu urusan domestik mereka dan Xinjiang sebagai integritas teritorial m
text_id  n_annotators  toxicity
 2-1000             3         1
 1-3665             3         0
Teks : ,Dalam beberapa pidatonya juru bicara pejuang Palestina, Abu Ubaidah menyampaikan rasa terima kasihn
text_id  n_annotators  toxicity
  3-376             1         0
 2-7590             1         0


### Hapus salinan yang benar-benar identik

Salinan dihapus hanya jika **teks, annotator, DAN keenam labelnya** (toxicity + 5 jenis) sama, karena tidak menambah informasi.
Salinan dengan annotator berbeda **tetap disimpan**, karena penilaian annotator lain (beserta domisilinya) dipakai model metadata.
Split per kelompok tetap menjaga agar salinan yang tersisa tidak bocor ke test.


In [34]:
kolom_label = ["toxicity"] + LABEL_COLS

# daftar annotator diurutkan supaya ['7','15'] dan ['15','7'] dianggap sama
# (re.findall dipakai karena ada beberapa baris annotators_id yang formatnya rusak, mis. "['20', '5', 33']")
df["daftar_annotator"] = df["annotators_id"].apply(
    lambda x: ",".join(sorted(set(re.findall(r"\d+", str(x))), key=int)))

sebelum = len(df)
df = (df.drop_duplicates(subset=["group_id", "daftar_annotator"] + kolom_label)
        .reset_index(drop=True))

print(f"Teks sebelum : {sebelum}")
print(f"Teks sesudah : {len(df)}  (dihapus {sebelum - len(df)} salinan identik)")
print(df["toxicity"].value_counts())

Teks sebelum : 26379
Teks sesudah : 26284  (dihapus 95 salinan identik)
toxicity
0    23887
1     2397
Name: count, dtype: int64


## Stage 1: balancing 1:3 lalu split per kelompok

- Balancing **caranya sama dengan sebelumnya** (rasio 1:3, seed sama). Karena ada salinan identik yang dihapus,
  teks non-toxic yang terambil bisa berbeda dari `splits.csv` lama.
- Yang di-split adalah **kelompok**, bukan teks. Label kelompok = 1 jika ada salinan yang toxic (dipakai untuk stratifikasi).
- Test 20%, lalu val 12,5% dari sisa (≈ 70/10/20).


In [35]:
pos = df[df["toxicity"] == 1]
neg = df[df["toxicity"] == 0].sample(n=3 * len(pos), random_state=SEED)
pool_s1 = pd.concat([pos, neg])

# 1 baris per kelompok
kelompok_s1 = pool_s1.groupby("group_id")["toxicity"].max().reset_index()

g_train_full, g_test = train_test_split(kelompok_s1, test_size=0.2,
                                        stratify=kelompok_s1["toxicity"], random_state=SEED)
g_train, g_val = train_test_split(g_train_full, test_size=0.125,
                                  stratify=g_train_full["toxicity"], random_state=SEED)

# kembalikan dari kelompok ke teks
split_kelompok = {}
for nama, g in [("train", g_train), ("val", g_val), ("test", g_test)]:
    for gid in g["group_id"]:
        split_kelompok[gid] = nama
pool_s1["s1_split"] = pool_s1["group_id"].map(split_kelompok)

print(pool_s1["s1_split"].value_counts())
print(pd.crosstab(pool_s1["s1_split"], pool_s1["toxicity"], normalize="index").round(3))

s1_split
train    6717
test     1910
val       961
Name: count, dtype: int64
toxicity      0      1
s1_split              
test      0.748  0.252
train     0.751  0.249
val       0.748  0.252


## Stage 2: teks toxic, split multi-label per kelompok

Label kelompok = 1 untuk suatu jenis toxicity jika ada salinan yang memiliki label tersebut.


In [36]:
pool_s2 = df[df["toxicity"] == 1].copy()

kelompok_s2 = pool_s2.groupby("group_id")[LABEL_COLS].max().reset_index()

def ml_split(data, test_size):
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    idx_train, idx_test = next(msss.split(data, data[LABEL_COLS].values))
    return data.iloc[idx_train], data.iloc[idx_test]

g2_train_full, g2_test = ml_split(kelompok_s2, 0.2)
g2_train, g2_val = ml_split(g2_train_full, 0.125)

split_kelompok_s2 = {}
for nama, g in [("train", g2_train), ("val", g2_val), ("test", g2_test)]:
    for gid in g["group_id"]:
        split_kelompok_s2[gid] = nama
pool_s2["s2_split"] = pool_s2["group_id"].map(split_kelompok_s2)

print(pool_s2["s2_split"].value_counts())
print(pool_s2.groupby("s2_split")[LABEL_COLS].sum().T)

s2_split
train    1677
test      482
val       238
Name: count, dtype: int64
s2_split                       test  train  val
insults                         214    736  104
identity_attack                 200    694   99
threat_incitement_to_violence    22     75   11
profanity_obscenity              73    250   36
sexually_explicit                12     43    6


In [37]:
splits = (df[["text_id", "group_id", "toxicity", "n_annotators"]]
          .rename(columns={"toxicity": "toxicity_majority"})
          .merge(pool_s1[["text_id", "s1_split"]], on="text_id", how="left")
          .merge(pool_s2[["text_id", "s2_split"]], on="text_id", how="left"))
splits = splits[splits["s1_split"].notna() | splits["s2_split"].notna()]

splits.to_csv(OUT_FILE, index=False)
print("Tersimpan:", OUT_FILE, "|", len(splits), "teks")

Tersimpan: /content/splits_grouped.csv | 9588 teks


## Cek hasil

1. Setiap kelompok hanya ada di **satu** split.
2. Tidak ada teks test yang isinya sama dengan teks train (dibandingkan dengan split lama).


In [38]:
cek = splits.merge(df[["text_id", "text", "input_model"]], on="text_id")

for stage in ["s1_split", "s2_split"]:
    s = cek.dropna(subset=[stage])
    split_per_kelompok = s.groupby("group_id")[stage].nunique()
    print(f"{stage}: kelompok yang tersebar di >1 split = {(split_per_kelompok > 1).sum()}")

s1_split: kelompok yang tersebar di >1 split = 0
s2_split: kelompok yang tersebar di >1 split = 0


In [39]:
def hitung_bocor(data, stage):
    s = data.dropna(subset=[stage])
    train = s[s[stage] == "train"]
    test  = s[s[stage] == "test"]
    bocor_asli  = test["text"].str.strip().isin(set(train["text"].str.strip())).sum()
    bocor_model = test["input_model"].isin(set(train["input_model"])).sum()
    return len(test), bocor_asli, bocor_model

lama = (pd.read_csv(OLD_SPLIT_FILE, dtype={"text_id": str})
          .merge(df[["text_id", "text", "input_model"]], on="text_id"))

hasil = []
for stage in ["s1_split", "s2_split"]:
    for nama, data in [("lama (splits.csv)", lama), ("baru (splits_grouped.csv)", cek)]:
        n_test, b_asli, b_model = hitung_bocor(data, stage)
        hasil.append({"stage": stage, "split": nama, "jumlah test": n_test,
                      "bocor (teks asli)": b_asli, "bocor (input model)": b_model})
pd.DataFrame(hasil)

,stage,split,jumlah test,bocor (teks asli),bocor (input model)
0,s1_split,lama (splits.csv),1913,90,25
1,s1_split,baru (splits_grouped.csv),1910,0,0
2,s2_split,lama (splits.csv),479,30,8
3,s2_split,baru (splits_grouped.csv),482,0,0
